# Data Merging & Feature Engineering

**Inputs:**  
- `data/processed/flight_features.csv` — 6.96M rows, 9 columns (from Student 1)  
- `data/processed/weather_features.csv` — hourly weather per airport per date  

**Output:** `data/processed/merged_dataset.csv` — single ready-to-preprocess dataset  

**Steps:**
1. Load both processed datasets
2. Prepare hourly weather (one row per airport, calendar date, hour)
3. Left-join flights ← weather on `(origin, date, dep_hour)` using flight `fl_date` / `date` aligned with weather `date`
4. Report & impute missing weather values
5. Cap outliers on numerical columns (IQR)
6. Final checks and save

In [40]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs/reports', exist_ok=True)

## 1 — Load Processed Datasets

In [41]:
flights = pd.read_csv('../data/processed/flight_features.csv')
weather_hourly = pd.read_csv('../data/processed/weather_features.csv')

print('=== Flights ===')
print(f'Shape: {flights.shape}')
print(f'Columns: {list(flights.columns)}')
print(flights.head(3))

print('\n=== Weather (hourly) ===')
print(f'Shape: {weather_hourly.shape}')
print(f'Columns: {list(weather_hourly.columns)}')
print(weather_hourly.head(3))

=== Flights ===
Shape: (6965267, 11)
Columns: ['airline', 'origin', 'dest', 'distance', 'dep_hour', 'day_of_week', 'month', 'is_weekend', 'fl_date', 'origin_state_nm', 'is_delayed']
  airline origin dest  distance  dep_hour  day_of_week  month  is_weekend  \
0      9E    JFK  DTW     509.0        12            1      1           0   
1      9E    MSP  CLE     622.0        10            1      1           0   
2      9E    JFK  RIC     288.0        14            1      1           0   

      fl_date origin_state_nm  is_delayed  
0  2024-01-01        New York           0  
1  2024-01-01       Minnesota           0  
2  2024-01-01        New York           0  

=== Weather (hourly) ===
Shape: (131760, 7)
Columns: ['iata', 'date', 'hour', 'precipitation', 'temperature_c', 'humidity_pct', 'wind_speed_kmh']
  iata        date  hour  precipitation  temperature_c  humidity_pct  \
0  DAL  2024-01-01     0            0.0            7.4          64.0   
1  DAL  2024-01-01     1            0.0   

## 2 — Prepare Weather for Merge

Weather is already hourly per **(airport IATA, calendar date, hour)** in `weather_features.csv`.  
Flights carry **`fl_date`** (or a precomputed **`date`**) plus **`dep_hour`**.  
We normalize both sides to the same ISO date string (`YYYY-MM-DD`) and join on **`origin`**, **`date`**, **`dep_hour`** so each row matches the weather at the origin on that local day and hour.

In [42]:
# Parse dates (hourly file is one row per airport / calendar day / hour)
weather_hourly['date'] = pd.to_datetime(weather_hourly['date'])

print(f'Date range: {weather_hourly["date"].min()} – {weather_hourly["date"].max()}')
print(f'Airports covered: {sorted(weather_hourly["iata"].unique())}')
print(f'Total airports: {weather_hourly["iata"].nunique()}')

Date range: 2024-01-01 00:00:00 – 2024-12-31 00:00:00
Airports covered: ['DAL', 'DFW', 'EWR', 'HOU', 'IAH', 'JFK', 'LAX', 'LGA', 'MDW', 'ORD', 'PHL', 'PHX', 'SAN', 'SAT', 'SJC']
Total airports: 15


In [43]:
# One row per (airport, calendar date, hour); mean if duplicate keys exist
weather_monthly = (
    weather_hourly
    .groupby(['iata', 'date', 'hour'], as_index=False)
    .agg(
        precipitation  = ('precipitation',  'mean'),
        temperature_c  = ('temperature_c',  'mean'),
        humidity_pct   = ('humidity_pct',   'mean'),
        wind_speed_kmh = ('wind_speed_kmh', 'mean')
    )
)

# Rename to match flight column names before join
weather_monthly.rename(columns={'iata': 'origin', 'hour': 'dep_hour'}, inplace=True)

print(f'Weather monthly profile shape: {weather_monthly.shape}')
print(f'Unique airports: {weather_monthly["origin"].nunique()}')
print(weather_monthly.head())

Weather monthly profile shape: (131760, 7)
Unique airports: 15
  origin       date  dep_hour  precipitation  temperature_c  humidity_pct  \
0    DAL 2024-01-01         0            0.0            7.4          64.0   
1    DAL 2024-01-01         1            0.0            6.2          68.0   
2    DAL 2024-01-01         2            0.0            5.1          71.0   
3    DAL 2024-01-01         3            0.0            3.9          73.0   
4    DAL 2024-01-01         4            0.0            3.2          74.0   

   wind_speed_kmh  
0            21.4  
1            22.5  
2            23.3  
3            22.1  
4            21.1  


## 3 — Merge Flights ← Weather

In [44]:
# Flight table: BTS `fl_date` or pipeline `date` — align with weather `date` as ISO strings
flights = flights.copy()
flights['date'] = pd.to_datetime(flights['fl_date']).dt.strftime('%Y-%m-%d')

weather_monthly = weather_monthly.copy()
weather_monthly['date'] = pd.to_datetime(weather_monthly['date']).dt.strftime('%Y-%m-%d')

# Left join: every flight row is kept; weather columns are NaN when origin not in weather data
merged = flights.merge(
    weather_monthly,
    on=['origin', 'date', 'dep_hour'],
    how='left'
)

print(f'Flights in:  {len(flights):,}')
print(f'Merged out:  {len(merged):,}')
print(f'Columns:     {list(merged.columns)}')

print(merged.head())

Flights in:  6,965,267
Merged out:  6,965,267
Columns:     ['airline', 'origin', 'dest', 'distance', 'dep_hour', 'day_of_week', 'month', 'is_weekend', 'fl_date', 'origin_state_nm', 'is_delayed', 'date', 'precipitation', 'temperature_c', 'humidity_pct', 'wind_speed_kmh']
  airline origin dest  distance  dep_hour  day_of_week  month  is_weekend  \
0      9E    JFK  DTW     509.0        12            1      1           0   
1      9E    MSP  CLE     622.0        10            1      1           0   
2      9E    JFK  RIC     288.0        14            1      1           0   
3      9E    RIC  JFK     288.0        16            1      1           0   
4      9E    DTW  MKE     237.0        10            1      1           0   

      fl_date origin_state_nm  is_delayed        date  precipitation  \
0  2024-01-01        New York           0  2024-01-01            0.0   
1  2024-01-01       Minnesota           0  2024-01-01            NaN   
2  2024-01-01        New York           0  2024-01

In [45]:
# --- Coverage report ---
weather_cols = ['precipitation', 'temperature_c', 'humidity_pct', 'wind_speed_kmh']

n_matched   = merged['precipitation'].notna().sum()
n_total     = len(merged)
pct_matched = n_matched / n_total * 100

print(f'Flights WITH weather data   : {n_matched:>10,}  ({pct_matched:.1f}%)')
print(f'Flights WITHOUT weather data: {n_total - n_matched:>10,}  ({100 - pct_matched:.1f}%)')
print()

# Which origin airports matched?
covered_airports = weather_monthly['origin'].unique()
all_origins      = flights['origin'].unique()
print(f'Origin airports in flight data : {len(all_origins)}')
print(f'Origin airports with weather   : {len(covered_airports)}')
print(f'Covered airports: {sorted(covered_airports)}')


Flights WITH weather data   :  1,973,141  (28.3%)
Flights WITHOUT weather data:  4,992,126  (71.7%)

Origin airports in flight data : 348
Origin airports with weather   : 15
Covered airports: ['DAL', 'DFW', 'EWR', 'HOU', 'IAH', 'JFK', 'LAX', 'LGA', 'MDW', 'ORD', 'PHL', 'PHX', 'SAN', 'SAT', 'SJC']


## 4 — Impute Missing Weather Values

Flights whose origin is not in the weather dataset get **NaN** after the join. We fill those using the **mean of that weather variable for the same calendar `date`**, computed across all rows that *do* have a real match that day (so each day gets its own typical conditions). If a date somehow had no non-missing values, we fall back to the **pre-imputation global mean** for that column.

In [46]:
print('=== Missing values BEFORE imputation ===')
print(merged[weather_cols].isnull().sum().to_string())

# Fallback if a date has no observed values at all for a column (rare)
fallback_global = {col: merged[col].mean() for col in weather_cols}

print('\n=== Per-date mean of *observed* values (used to fill NaNs) ===')
for col in weather_cols:
    dm = merged.groupby('date', observed=True)[col].mean()
    print(f'  {col:<20}: across dates: min={dm.min():.4f}  max={dm.max():.4f}  (n_dates={len(dm)})')

for col in weather_cols:
    daily_mean = merged.groupby('date', observed=True)[col].transform('mean')
    merged[col] = merged[col].fillna(daily_mean)

for col in weather_cols:
    if merged[col].isna().any():
        merged[col] = merged[col].fillna(fallback_global[col])

print('\n=== Missing values AFTER imputation ===')
print(merged[weather_cols].isnull().sum().to_string())

=== Missing values BEFORE imputation ===
precipitation     4992126
temperature_c     4992126
humidity_pct      4992126
wind_speed_kmh    4992126

=== Per-date mean of *observed* values (used to fill NaNs) ===
  precipitation       : across dates: min=0.0000  max=0.9119  (n_dates=366)
  temperature_c       : across dates: min=-2.4396  max=28.4487  (n_dates=366)
  humidity_pct        : across dates: min=45.3504  max=91.6722  (n_dates=366)
  wind_speed_kmh      : across dates: min=5.9998  max=22.2605  (n_dates=366)

=== Missing values AFTER imputation ===
precipitation     0
temperature_c     0
humidity_pct      0
wind_speed_kmh    0


## 5 — Outlier Handling (IQR Capping)

We **cap** (winsorise) rather than drop outliers to preserve dataset size.  
Values below Q1 − 1.5×IQR are set to that lower bound; values above Q3 + 1.5×IQR are set to the upper bound.

In [47]:
def cap_outliers_iqr(df, col, multiplier=1.5):
    """Winsorise a column using the IQR method. Returns (n_capped, lower, upper)."""
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0, q1, q3
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    mask  = (df[col] < lower) | (df[col] > upper)
    n_capped = int(mask.sum())
    df[col]  = df[col].clip(lower=lower, upper=upper)
    return n_capped, lower, upper


# Numerical columns that may carry outliers worth capping
num_cols_to_cap = ['distance', 'precipitation', 'temperature_c', 'humidity_pct', 'wind_speed_kmh']

print(f'{"Column":<22} {"Capped":>10} {"Lower":>10} {"Upper":>10}')
print('-' * 56)
for col in num_cols_to_cap:
    n, lo, hi = cap_outliers_iqr(merged, col)
    pct = n / len(merged) * 100
    print(f'{col:<22} {n:>8,} ({pct:.2f}%)   [{lo:.2f}, {hi:.2f}]')

Column                     Capped      Lower      Upper
--------------------------------------------------------
distance                413,198 (5.93%)   [-606.00, 2074.00]
precipitation           594,696 (8.54%)   [-0.18, 0.31]
temperature_c            10,141 (0.15%)   [-9.31, 45.11]
humidity_pct            701,862 (10.08%)   [40.37, 87.80]
wind_speed_kmh          370,215 (5.32%)   [3.26, 21.24]


## 6 — Final Dataset Inspection

In [48]:
print(f'Final shape : {merged.shape}')
print(f'\nColumns     : {list(merged.columns)}')
print(f'\nMissing values:\n{merged.isnull().sum().to_string()}')
print(f'\nTarget distribution:')
vc = merged['is_delayed'].value_counts()
for label, count in vc.items():
    print(f'  {label} → {count:,} ({count/len(merged)*100:.2f}%)')

Final shape : (6965267, 16)

Columns     : ['airline', 'origin', 'dest', 'distance', 'dep_hour', 'day_of_week', 'month', 'is_weekend', 'fl_date', 'origin_state_nm', 'is_delayed', 'date', 'precipitation', 'temperature_c', 'humidity_pct', 'wind_speed_kmh']

Missing values:
airline            0
origin             0
dest               0
distance           0
dep_hour           0
day_of_week        0
month              0
is_weekend         0
fl_date            0
origin_state_nm    0
is_delayed         0
date               0
precipitation      0
temperature_c      0
humidity_pct       0
wind_speed_kmh     0

Target distribution:
  0 → 5,561,879 (79.85%)
  1 → 1,403,388 (20.15%)


In [49]:
print('=== Numerical Summary ===')
num_cols = ['distance', 'dep_hour', 'precipitation', 'temperature_c', 'humidity_pct', 'wind_speed_kmh']
print(merged[num_cols].describe().round(3).to_string())

=== Numerical Summary ===
          distance     dep_hour  precipitation  temperature_c  humidity_pct  wind_speed_kmh
count  6965267.000  6965267.000    6965267.000    6965267.000   6965267.000     6965267.000
mean       810.525       12.992          0.077         17.568        64.391          12.380
std        530.048        4.898          0.101          8.239        11.264           3.850
min         11.000        0.000          0.000         -9.305        40.375           3.259
25%        399.000        9.000          0.000         11.100        58.160          10.000
50%        680.000       13.000          0.021         18.200        64.474          12.309
75%       1069.000       17.000          0.122         24.703        70.017          14.494
max       2074.000       24.000          0.305         45.109        87.803          21.235


In [50]:
print(merged.head())

  airline origin dest  distance  dep_hour  day_of_week  month  is_weekend  \
0      9E    JFK  DTW     509.0        12            1      1           0   
1      9E    MSP  CLE     622.0        10            1      1           0   
2      9E    JFK  RIC     288.0        14            1      1           0   
3      9E    RIC  JFK     288.0        16            1      1           0   
4      9E    DTW  MKE     237.0        10            1      1           0   

      fl_date origin_state_nm  is_delayed        date  precipitation  \
0  2024-01-01        New York           0  2024-01-01       0.000000   
1  2024-01-01       Minnesota           0  2024-01-01       0.014526   
2  2024-01-01        New York           0  2024-01-01       0.000000   
3  2024-01-01        Virginia           0  2024-01-01       0.014526   
4  2024-01-01        Michigan           0  2024-01-01       0.014526   

   temperature_c  humidity_pct  wind_speed_kmh  
0       5.300000     71.000000        6.800000  
1     

## 7 — Save Merged Dataset

In [51]:
output_path = '../data/processed/merged_dataset.csv'
merged.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Shape: {merged.shape}')
print(f'\nFinal columns:')
for i, col in enumerate(merged.columns, 1):
    dtype = str(merged[col].dtype)
    print(f'  {i:2}. {col:<22} ({dtype})')

Saved: ../data/processed/merged_dataset.csv
Shape: (6965267, 16)

Final columns:
   1. airline                (str)
   2. origin                 (str)
   3. dest                   (str)
   4. distance               (float64)
   5. dep_hour               (int64)
   6. day_of_week            (int64)
   7. month                  (int64)
   8. is_weekend             (int64)
   9. fl_date                (str)
  10. origin_state_nm        (str)
  11. is_delayed             (int64)
  12. date                   (str)
  13. precipitation          (float64)
  14. temperature_c          (float64)
  15. humidity_pct           (float64)
  16. wind_speed_kmh         (float64)
